In this notebook we implement pybdshadow for calculating building's shadowing in bologna's city for adressing a Urban Thermal Adaptability index 

In [1]:
!pip install pybdshadow


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\guada\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import geopandas as gpd
import pybdshadow
import matplotlib.pyplot as plt
import json
import time
import os
from tqdm import tqdm
from pyproj import CRS
import numpy as np

import warnings
warnings.filterwarnings("ignore", message="CRS not set for some of the concatenation inputs")
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
num_parts = 6 # number of files in clean_results (the number comes from 2.4.2.3.trees-vector-preparation.ipynb)
input_folder = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors-clean/'
output_folder = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-shadows/'
os.makedirs(output_folder, exist_ok=True)


In [6]:
p = os.path.join(input_folder, f"part3_clean.geojson")
p = gpd.read_file(p)
p.head()

,height,filename,geometry
0,1,meta_canopy_tile_10.tif,"POLYGON ((691456.167 4930003.733, 691456.631 4..."
1,1,meta_canopy_tile_10.tif,"POLYGON ((691463.398 4930004.883, 691464.547 4..."
2,3,meta_canopy_tile_10.tif,"POLYGON ((691454.186 4930008.848, 691454.73 49..."
3,1,meta_canopy_tile_10.tif,"POLYGON ((691466.266 4930005.883, 691467.031 4..."
4,4,meta_canopy_tile_10.tif,"POLYGON ((691465.398 4930006.883, 691466.547 4..."


In [ ]:
# computing shadows 
# THIS BLOCK TAKES AROUND 1H20' TO FINISH. 
'''

date = pd.to_datetime('2024-08-09 09:58:10').tz_localize('UTC')

for j in tqdm(range(num_parts)):
    part_file = os.path.join(input_folder, f"part{j+1}_clean.geojson")
    if not os.path.exists(part_file):
        continue
        
    trees = gpd.read_file(part_file)
    trees = trees.to_crs(CRS.from_epsg(4326)) # pybdshadow needs EPSG:4326
    trees = pybdshadow.bd_preprocess(trees)

    # Divide each one in 5 parts for avoiding problems with RAM
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts):
        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(part, date, roof=False, include_building=True)
            print(f"Processing time part{j+1} subpart{k+1}: {round(time.time() - start, 2)} seconds")
            shadow_part.append(shadow_subpart)
        except Exception as e:
            print(f"Error in part{j+1} subpart{k+1}: {e}")
        time.sleep(10)
    
    # join results
    shadows = gpd.GeoDataFrame(pd.concat(shadow_part, ignore_index=True), crs=shadow_part[0].crs)
    # dissolve
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)

    output_path = os.path.join(output_folder, f"part{j}_shadow.geojson")
    #shadows.set_crs("EPSG:32632", inplace=True)
    shadows.to_file(output_path, driver="GeoJSON")
'''

  0%|          | 0/20 [00:00<?, ?it/s]

C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopandas\geodataframe.py:257: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  super().__setattr__(attr, val)
  0%|          | 0/20 [00:00<?, ?it/s]


ValueError: Assigning CRS to a GeoDataFrame without a geometry column is not supported. Use GeoDataFrame.set_geometry to set the active geometry column.

In [9]:
date = pd.to_datetime('2024-08-09 09:58:10').tz_localize('UTC')

for j in tqdm(range(num_parts)):
    part_file = os.path.join(input_folder, f"part{j+1}_clean.geojson")
    if not os.path.exists(part_file):
        continue

    # ----------------------------------------------------------------------
    # READ + CHECK EMPTY
    # ----------------------------------------------------------------------
    trees = gpd.read_file(part_file)
    
    # Skip if empty or geometry missing
    if trees.empty or trees.geometry.is_empty.all() or trees.geometry.isna().all():
        print(f"part{j+1}: EMPTY after read -> skipped")
        continue

    # ----------------------------------------------------------------------
    # CRS
    # ----------------------------------------------------------------------
    trees = trees.to_crs(CRS.from_epsg(4326))  # pybdshadow needs EPSG:4326

    # ----------------------------------------------------------------------
    # PREPROCESS + CHECK EMPTY
    # ----------------------------------------------------------------------
    trees = pybdshadow.bd_preprocess(trees)

    if trees.empty or trees.geometry.is_empty.all() or trees.geometry.isna().all():
        print(f"part{j+1}: EMPTY after bd_preprocess -> skipped")
        continue

    # ----------------------------------------------------------------------
    # SPLIT
    # ----------------------------------------------------------------------
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts):

        # Skip empty subpart
        if part.empty or part.geometry.is_empty.all() or part.geometry.isna().all():
            print(f"part{j+1} subpart{k+1}: EMPTY -> skipped")
            continue

        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(
                part, date, roof=False, include_building=True
            )
            print(f"Processing time part{j+1} subpart{k+1}: {round(time.time() - start, 2)}s")
            shadow_part.append(shadow_subpart)
        except Exception as e:
            print(f"Error in part{j+1} subpart{k+1}: {e}")

        time.sleep(10)

    # ----------------------------------------------------------------------
    # NO SUBPARTS → SKIP
    # ----------------------------------------------------------------------
    if not shadow_part:
        print(f"part{j+1}: No shadow parts produced -> skipped")
        continue

    # ----------------------------------------------------------------------
    # MERGE RESULTS
    # ----------------------------------------------------------------------
    shadows = gpd.GeoDataFrame(pd.concat(shadow_part, ignore_index=True),
                               crs=shadow_part[0].crs)

    # Skip if shadows empty
    if shadows.empty:
        print(f"part{j+1}: EMPTY shadows -> skipped")
        continue

    # ----------------------------------------------------------------------
    # DISSOLVE / UNION
    # ----------------------------------------------------------------------
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)

    # Skip if union empty
    if shadows.empty:
        print(f"part{j+1}: EMPTY union -> skipped")
        continue

    # ----------------------------------------------------------------------
    # SAVE OUTPUT
    # ----------------------------------------------------------------------
    output_path = os.path.join(output_folder, f"part{j}_shadow.geojson")
    shadows.to_file(output_path, driver="GeoJSON")


  0%|          | 0/20 [00:00<?, ?it/s]

part1: EMPTY after read -> skipped
part2: EMPTY after read -> skipped
Processing time part3 subpart1: 31.54s
Processing time part3 subpart2: 33.52s
Processing time part3 subpart3: 37.82s
Processing time part3 subpart4: 38.69s
Processing time part3 subpart5: 37.66s


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
 35%|███▌      | 7/20 [04:29<06:41, 30.90s/it]

part4: EMPTY after read -> skipped
part5: EMPTY after read -> skipped
part6: EMPTY after read -> skipped
part7: EMPTY after read -> skipped
part8: EMPTY after read -> skipped
part9: EMPTY after read -> skipped
part10: EMPTY after read -> skipped
Processing time part11 subpart1: 3.05s
Processing time part11 subpart2: 4.52s


 35%|███▌      | 7/20 [04:49<06:41, 30.90s/it]

Processing time part11 subpart3: 5.7s
Processing time part11 subpart4: 7.46s
Processing time part11 subpart5: 5.1s


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
 55%|█████▌    | 11/20 [05:50<03:51, 25.73s/it]

Error in part12 subpart1: Unable to allocate 3.53 GiB for an array with shape (473842320,) and data type int64
Error in part12 subpart2: Unable to allocate 5.04 GiB for an array with shape (98104, 6895) and data type int64
Error in part12 subpart3: Unable to allocate 1.64 GiB for an array with shape (439207131,) and data type int32
Processing time part12 subpart4: 404.44s
Error in part12 subpart5: Unable to allocate 2.86 GiB for an array with shape (384367554,) and data type int64


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
 60%|██████    | 12/20 [27:41<18:27, 138.44s/it]


KeyboardInterrupt: 

In [ ]:
# join all the results in one dataframe

input_folder = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-shadows/'
total_shadows = []

for j in tqdm(range(num_parts)):
    
    part_file = os.path.join(input_folder, f"part{j}_shadow.geojson") #correjir a j++1 aca y anterior bloque
    if not os.path.exists(part_file):
        continue
    shadow_part = gpd.read_file(part_file)
    total_shadows.append(shadow_part)

    # Unir y guardar
    shadows = gpd.GeoDataFrame(pd.concat(total_shadows, ignore_index=True), crs=total_shadows[0].crs)


shadows.to_file('../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/shadows_trees_2024-08-09_095810', driver="GeoJSON")